In [0]:
%pip install yfinance

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import yfinance as yf
import pandas as pd

# 50 tickers across sectors: Tech, Finance, Healthcare, Energy, Consumer, Industrial
tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "AMD",
    "JPM", "GS", "BAC", "MS", "BLK", "C", "WFC", "AXP",
    "JNJ", "UNH", "PFE", "MRK", "ABT", "TMO", "LLY", "ABBV",
    "XOM", "CVX", "COP", "SLB", "OXY", "NEE",
    "WMT", "PG", "KO", "PEP", "COST", "NKE", "MCD", "SBUX",
    "CAT", "HON", "UPS", "BA", "GE", "LMT", "RTX", "DE",
    "DIS", "NFLX", "V", "MA"
]

# Download 5 years of daily data
data = yf.download(tickers, period="5y", group_by="ticker")
print(f"Shape: {data.shape}")
print(f"Date range: {data.index.min()} to {data.index.max()}")


[*********************100%***********************]  50 of 50 completed

Shape: (1255, 250)
Date range: 2021-06-01 00:00:00 to 2026-05-29 00:00:00


In [0]:
# Reshape from wide format (1 row per day, 250 columns) 
# into long format (1 row per stock per day, clean columns)
records = []
for ticker in tickers:
    try:
        df = data[ticker].copy()
        df["Ticker"] = ticker
        df["Date"] = df.index
        records.append(df)
    except:
        print(f"Skipping {ticker}")

# Stack all stocks into one tall table and clean up
stock_df = pd.concat(records).reset_index(drop=True)
stock_df.columns = ["Open", "High", "Low", "Close", "Volume", "Ticker", "Date"]

# Reorder columns so Date and Ticker come first
stock_df = stock_df[["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]]

# Drop any rows with missing prices
stock_df = stock_df.dropna(subset=["Close"])

print(f"Total rows: {len(stock_df):,}")
print(f"Unique tickers: {stock_df['Ticker'].nunique()}")
print(stock_df.head(10))

Total rows: 62,750
Unique tickers: 50
        Date Ticker        Open        High         Low       Close    Volume
0 2021-06-01   AAPL  121.921971  122.185150  120.810754  121.142166  67637100
1 2021-06-02   AAPL  121.142176  122.077937  120.917987  121.902481  59278900
2 2021-06-03   AAPL  121.532077  121.697783  120.021208  120.420860  76229200
3 2021-06-04   AAPL  120.937462  122.974697  120.723015  122.711510  75169300
4 2021-06-07   AAPL  122.984455  123.130669  121.678291  122.721275  71057600
5 2021-06-08   AAPL  123.403597  125.216644  123.023445  123.540062  74403800
6 2021-06-09   AAPL  123.998201  124.524568  123.325620  123.920219  56877900
7 2021-06-10   AAPL  123.813000  124.953466  122.760274  122.925980  71186400
8 2021-06-11   AAPL  123.335355  124.222383  122.916211  124.134651  53522400
9 2021-06-14   AAPL  124.592809  127.244129  123.861745  127.185646  96906500


In [0]:
# Convert pandas DataFrame to a Spark DataFrame (Databricks runs on Spark)
spark_df = spark.createDataFrame(stock_df)

# Save it as a permanent table in your Databricks catalog
spark_df.write.mode("overwrite").saveAsTable("stock_prices_raw")

print("Table saved! Verifying...")
result = spark.sql("SELECT COUNT(*) as total_rows, COUNT(DISTINCT Ticker) as tickers FROM stock_prices_raw")
result.show()

Table saved! Verifying...
+----------+-------+
|total_rows|tickers|
+----------+-------+
|     62750|     50|
+----------+-------+



In [0]:
# To download Stock_analytics data

In [0]:
df = spark.sql("SELECT * FROM stock_analytics")

In [0]:
df.toPandas().to_csv("/tmp/stock_analytics_full.csv", index=False)

In [0]:
print(f"Exported {df.count()} rows")

Exported 62700 rows


In [0]:
display(spark.sql("SELECT * FROM stock_analytics"))

Date,Ticker,Open,High,Low,Close,Volume,Sector,prev_close,daily_return_pct
2021-06-02T00:00:00.000Z,AAPL,121.14217571934485,122.07793673418138,120.91798692698966,121.90248107910156,59278900,Technology,121.14216613769531,0.6276
2021-06-03T00:00:00.000Z,AAPL,121.5320769508381,121.69778300232346,120.02120840977952,120.42086029052734,76229200,Technology,121.90248107910156,-1.2154
2021-06-04T00:00:00.000Z,AAPL,120.93746163646239,122.97469686847107,120.72301503974921,122.71150970458984,75169300,Technology,120.42086029052734,1.9022
2021-06-07T00:00:00.000Z,AAPL,122.9844550910691,123.13066937489863,121.67829101219802,122.72127532958984,71057600,Technology,122.71150970458984,0.008
2021-06-08T00:00:00.000Z,AAPL,123.40359727053726,125.21664395698274,123.02344459916287,123.5400619506836,74403800,Technology,122.72127532958984,0.6672
2021-06-09T00:00:00.000Z,AAPL,123.99820136642403,124.52456834027448,123.32562010482052,123.92021942138672,56877900,Technology,123.5400619506836,0.3077
2021-06-10T00:00:00.000Z,AAPL,123.81300009816206,124.95346562999302,122.76027355328648,122.92597961425781,71186400,Technology,123.92021942138672,-0.8023
2021-06-11T00:00:00.000Z,AAPL,123.33535493510138,124.22238272093318,122.91621132668672,124.13465118408203,53522400,Technology,122.92597961425781,0.9833
2021-06-14T00:00:00.000Z,AAPL,124.59280941543645,127.24412880162201,123.86174536220003,127.1856460571289,96906500,Technology,124.13465118408203,2.4578
2021-06-15T00:00:00.000Z,AAPL,126.65927274152149,127.30261260822043,126.12315618593904,126.3668441772461,62746300,Technology,127.1856460571289,-0.6438
